# ==============================================================================
# 🚢 MARITIME EVACUATION SURVIVAL: TITANIC VIA STACKING CLASSIFIER
# ==============================================================================
### Core Theoretical Principles:
Stacked Generalization (Wolpert, 1992) learns to combine predictions from multiple base learners using a meta-learner:
$$\tilde{X} = [\hat{P}_1(y=1 \mid X), \hat{P}_2(y=1 \mid X), \dots, \hat{P}_M(y=1 \mid X)]$$
$$\hat{y} = g(\tilde{X})$$

To prevent catastrophic overfitting / target leakage, $\tilde{X}$ is generated using **Out-Of-Fold (OOF)** cross-validation predictions.


In [1]:
# STEP 1: ENTERPRISE ENVIRONMENT & REPRODUCIBILITY SEED
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Enterprise Stacking Environment initialized.")


✅ STEP 1: Enterprise Stacking Environment initialized.


## 📥 STEP 2 & 3: RAW DATASET INGESTION & 3-SECOND HEALTH AUDIT
Loading Titanic passenger manifest (891 passenger records).


In [2]:
# STEP 2 & 3: INGESTION & AUDIT
data_path = 'data/titanic/Titanic-Dataset.csv'
df = pd.read_csv(data_path)

print(f"📊 Dataset Dimensions: {df.shape[0]} passengers, {df.shape[1]} attributes")
print(f"Survival Distribution:\n{df['Survived'].value_counts(normalize=True).round(3)}")
df.head(3)


📊 Dataset Dimensions: 891 passengers, 12 attributes
Survival Distribution:
Survived
0    0.616
1    0.384
Name: proportion, dtype: float64


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


## 🧹 STEP 4 & 5: STRICT SEGREGATION & HYGIENE
Separating features and target `Survived`.


In [3]:
# STEP 4 & 5: SEGREGATION & HYGIENE
X = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']].copy()
y = df['Survived'].copy()

print(f"Features: {X.columns.tolist()}")


Features: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']


## 🔒 STEP 6: ZERO-DATA-LEAKAGE STRATIFIED SPLIT
Stratified 80/20 train/test split.


In [4]:
# STEP 6: STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Train Cohort: {X_train.shape[0]} passengers")
print(f"Test Cohort : {X_test.shape[0]} passengers")


Train Cohort: 712 passengers
Test Cohort : 179 passengers


## ⚙️ STEP 7: PREPROCESSOR PIPELINE
Standardizing numerical indicators and encoding categorical attributes.


In [5]:
# STEP 7: PREPROCESSOR PIPELINE
num_cols = ['Age', 'Fare', 'Pclass', 'SibSp', 'Parch']
cat_cols = ['Sex', 'Embarked']

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ]), cat_cols)
])
print("✅ STEP 7: Preprocessor ready.")


✅ STEP 7: Preprocessor ready.


## ⚠️ STEP 8: BASELINE STACKING FIT
Fitting StackingClassifier combining Logistic Regression, Random Forest, and Gaussian Naive Bayes with a Logistic Meta-Learner.


In [6]:
# STEP 8: BASELINE STACKING FIT
base_estimators = [
    ('lr', LogisticRegression(random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)),
    ('nb', GaussianNB())
]

stack_clf = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(random_state=42),
    cv=5,
    n_jobs=-1
)

pipe_stack = Pipeline([('prep', preprocessor), ('stack', stack_clf)])
pipe_stack.fit(X_train, y_train)

acc_stack = pipe_stack.score(X_test, y_test)
print(f"Baseline Stacking Accuracy: {acc_stack * 100:.2f}%")


Baseline Stacking Accuracy: 82.12%


## 🎯 STEP 9: TUNING META-LEARNER VIA 5-FOLD CV
Optimizing meta-learner regularization `final_estimator__C` and `passthrough`.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'stack__final_estimator__C': [0.1, 1.0, 10.0],
    'stack__passthrough': [False, True]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(pipe_stack, param_grid=param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Meta-Learner Parameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV ROC-AUC: {grid.best_score_*100:.2f}%")


🏆 Best Meta-Learner Parameters: {'stack__final_estimator__C': 10.0, 'stack__passthrough': False}
🎯 Best 5-Fold CV ROC-AUC: 87.01%


## 📊 STEP 10 & 11: EVALUATION REPORT & DIAGNOSTICS
Comprehensive classification report and confusion matrix.


In [8]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("📋 MARITIME SAFETY STACKING REPORT:")
print(classification_report(y_test, y_pred, target_names=['Perished', 'Survived']))
print(f"Test ROC-AUC Score : {roc_auc_score(y_test, y_prob):.4f}")
print(f"Test Set Accuracy  : {accuracy_score(y_test, y_pred)*100:.2f}%")


📋 MARITIME SAFETY STACKING REPORT:
              precision    recall  f1-score   support

    Perished       0.81      0.92      0.86       110
    Survived       0.83      0.65      0.73        69

    accuracy                           0.82       179
   macro avg       0.82      0.79      0.80       179
weighted avg       0.82      0.82      0.81       179

Test ROC-AUC Score : 0.8451
Test Set Accuracy  : 81.56%


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & SMOKE TEST
Deploying pipeline and validating ensemble latency.


In [9]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/titanic_stacking_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_passenger = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_class = loaded_pipe.predict(sample_passenger)[0]
pred_prob = loaded_pipe.predict_proba(sample_passenger)[0, 1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🚢 LIVE PASSENGER STACKED INFERENCE:")
print(f"   Outcome: {'✅ SURVIVED' if pred_class == 1 else '❌ PERISHED'}")
print(f"   Stacked Probability : {pred_prob * 100:.2f}%")
print(f"   Inference Latency   : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/titanic_stacking_pipeline.joblib (69,288 bytes)



🚢 LIVE PASSENGER STACKED INFERENCE:
   Outcome: ❌ PERISHED
   Stacked Probability : 12.06%
   Inference Latency   : 72.087 ms (SLA < 2.0ms: CHECK)
